In [22]:
!git clone https://github.com/Prithviraj-chw/FlyRank-Starter.git
%cd FlyRank-Starter
!ls data/raw/


Cloning into 'FlyRank-Starter'...
remote: Enumerating objects: 108, done.
remote: Counting objects: 100% (108/108), done.
remote: Compressing objects: 100% (79/79), done.
remote: Total 108 (delta 28), reused 77 (delta 13), pack-reused 0 (from 0)
Receiving objects: 100% (108/108), 1.84 MiB | 10.54 MiB/s, done.
Resolving deltas: 100% (28/28), done.
/content/FlyRank-Starter/FlyRank-Starter/FlyRank-Starter
content_refresh_anonymized.csv


In [23]:
import pandas as pd

DATA_PATH = "data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(DATA_PATH)

# ML-03 — Frame Your Lane as an ML Task

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

Lane: Refresh / Content Opportunity Scoring.

 ML task type: Scoring / Ranking.

I'm not sorting pages into fixed buckets (classification) or finding natural groups (clustering). I'm ordering ~30k pages by "how worth refreshing is this, right now" so an editor can work top-down. That's a ranking/scoring problem, not a yes/no classification problem.

In [24]:
lane = "Refresh / Content Opportunity Scoring"
task_type = "Scoring / Ranking"

print(f"Lane: {lane}")
print(f"ML task type: {task_type}")

Lane: Refresh / Content Opportunity Scoring
ML task type: Scoring / Ranking


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

This is a proxy, not a directly observed target. There's no column that literally says "needs refresh" or "is opportunity" — nobody hand-labeled that. So I'm using trend_pct (the observed % change in performance over the last 90 days) combined with trend_direction (down/stable/up) as a stand-in for "this page needs attention." A page is a refresh opportunity if it's declining (trend_direction == "down", large negative trend_pct) while still having real search demand (search_volume, impressions_90d not near zero). It's the best available approximation from real 90-day performance data — not a hand-written rule like content age.

In [25]:
target_definition = (
    "This is a proxy, not a directly observed target. No column labels "
    "'needs refresh' directly, so I'm using trend_pct (observed % change in "
    "performance over 90 days) combined with trend_direction as a stand-in. "
    "A page is a refresh opportunity if it's declining while still having "
    "real search demand (search_volume, impressions_90d)."
)
print(target_definition)

opportunity_candidates = df[
    (df["trend_direction"] == "down") & (df["search_volume"] > 0)
]
print(f"Candidate refresh-worthy pages: {len(opportunity_candidates)} of {len(df)}")

This is a proxy, not a directly observed target. No column labels 'needs refresh' directly, so I'm using trend_pct (observed % change in performance over 90 days) combined with trend_direction as a stand-in. A page is a refresh opportunity if it's declining while still having real search demand (search_volume, impressions_90d).
Candidate refresh-worthy pages: 8517 of 30000


## 3. Success metric

*One metric you can defend. What number means 'good'?*

Success metric: Precision@50 — of the top 50 pages my score ranks highest, what % are actually declining (trend_direction == "down") with meaningful search_volume/impressions_90d? Editors work a limited list per sprint, so what matters is whether the top of the queue is genuinely worth their time, not accuracy across all 30,000 pages.

In [26]:
success_metric = "Precision@50: of the top 50 ranked pages, % that are truly declining with real search demand"
print(f"Success metric: {success_metric}")

Success metric: Precision@50: of the top 50 ranked pages, % that are truly declining with real search demand


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

This dataset (content_refresh_anonymized.csv) is already my lane's slice — Refresh/Content Opportunity Scoring. One row = one content page (identified by content_id), with columns describing its search demand, current performance, and 90-day trend.

In [27]:
print(f"Shape: {df.shape[0]} rows x {df.shape[1]} columns")
print(list(df.columns))
df.head()

Shape: 30000 rows x 44 columns
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [28]:
# Confirming whether the unit of analysisis : content_id is actually unique per row
print(f"\nUnique content_id values: {df['content_id'].nunique()} of {len(df)} rows")
print("One row = one content page" if df['content_id'].nunique() == len(df) else "WARNING: content_id has duplicates -- unit of analysis isn't clean")


Unique content_id values: 30000 of 30000 rows
One row = one content page


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A fixed rule like "refresh anything with trend_direction == down" would flag every declining page equally — including ones with zero search demand that aren't worth anyone's time . Real opportunity depends on several signals interacting at once: decline magnitude (trend_pct), how much demand still exists (search_volume, impressions_90d, impression_tier), current visibility (avg_position, position_tier), and content quality signals (word_count, engagement_rate, scroll_rate). A simple if-statement can't weigh these together or learn which combinations actually separate a worthwhile refresh from a dead page. A model can.

In [29]:
reasons = """
A fixed rule ("refresh anything trending down") would flag every declining page
equally -- including ones with zero real search demand. Row 2 in this dataset
is a clear example: search_volume = 0.0, trend_pct = -60.9, but there's no
demand to recover.

Real opportunity depends on signals interacting: decline magnitude (trend_pct),
demand still present (search_volume, impressions_90d), current visibility
(avg_position, position_tier), and content quality (word_count,
engagement_rate, scroll_rate). A model can learn which combinations actually
separate a worthwhile refresh from a dead page -- a single if-statement can't.
"""
print(reasons)

# quick illustration: a rule alone misses this distinction
rule_flagged = df[df["trend_direction"] == "down"]
zero_demand_flagged = rule_flagged[rule_flagged["search_volume"] == 0]
print(f"Rule flags {len(rule_flagged)} pages, {len(zero_demand_flagged)} of which have zero search demand")


A fixed rule ("refresh anything trending down") would flag every declining page
equally -- including ones with zero real search demand. Row 2 in this dataset
is a clear example: search_volume = 0.0, trend_pct = -60.9, but there's no
demand to recover.

Real opportunity depends on signals interacting: decline magnitude (trend_pct),
demand still present (search_volume, impressions_90d), current visibility
(avg_position, position_tier), and content quality (word_count,
engagement_rate, scroll_rate). A model can learn which combinations actually
separate a worthwhile refresh from a dead page -- a single if-statement can't.

Rule flags 16262 pages, 7008 of which have zero search demand


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.